In [3]:
# Imports and setup
import fsspec
import pyarrow.parquet as pq
import pyarrow as pa

import os
import time

print("All libraries imported successfully!")

All libraries imported successfully!


## Remote Schema expolaration

In [6]:
# Helper functions to get stats.
def get_detailed_column_stats(URL, column_names=None):
    """Get detailed statistics for specific columns or all columns"""
    
    with fsspec.open(URL, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow
        
        # If single column provided as string, convert to list
        if isinstance(column_names, str):
            column_names = [column_names]
        
        # Validate columns exist
        valid_columns = []
        for col_name in column_names:
            if col_name in schema.names:
                valid_columns.append(col_name)
            else:
                print(f"Warning: Column '{col_name}' not found in schema. Available columns: {schema.names}")
        
        if not valid_columns:
            print("No valid columns to analyze")
            return
        
        print(f"=== ANALYZING {len(valid_columns)} COLUMN(S) ===")
        
        # Process each column
        for column_name in valid_columns:
            col_idx = schema.names.index(column_name)
            col_type = schema.field(column_name).type
            print(f"\n{'='*50}")
            print(f"DETAILED STATS FOR '{column_name}'")
            print(f"{'='*50}")
            
            total_nulls = 0
            total_values = 0
            has_any_stats = False
            
            for rg_idx in range(metadata.num_row_groups):
                rg_metadata = metadata.row_group(rg_idx)
                col_metadata = rg_metadata.column(col_idx)
                rg_num_rows = rg_metadata.num_rows
                
                if col_metadata.statistics:
                    has_any_stats = True
                    stats = col_metadata.statistics
                    
                    # Print row group header only if we have stats to show
                    print(f"\nRow Group {rg_idx} (Rows: {rg_num_rows:,}):")
                    
                    # Size information
                    if hasattr(col_metadata, 'total_compressed_size'):
                        print(f"  Compressed size: {col_metadata.total_compressed_size:,} bytes")
                    if hasattr(col_metadata, 'total_uncompressed_size'):
                        print(f"  Uncompressed size: {col_metadata.total_uncompressed_size:,} bytes")
                    
                    # Null statistics
                    if stats.has_null_count:
                        null_count = stats.null_count
                        total_nulls += null_count
                        null_percentage = (null_count / rg_num_rows) * 100 if rg_num_rows > 0 else 0
                        print(f"  Null count: {null_count:,} ({null_percentage:.1f}%)")
                    
                    # Min/Max values - with proper overflow handling
                    if stats.has_min_max:
                        try:
                            # Try to get as proper Python objects
                            min_val = stats.min
                            max_val = stats.max
                            
                            # Handle different data types
                            if pa.types.is_timestamp(col_type):
                                # For timestamp types
                                min_dt = pa.scalar(min_val).cast(col_type).as_py()
                                max_dt = pa.scalar(max_val).cast(col_type).as_py()
                                print(f"  Min: {min_dt}")
                                print(f"  Max: {max_dt}")
                            elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                                # For string types, show first/last few characters if long
                                min_str = str(min_val)
                                max_str = str(max_val)
                                if len(min_str) > 50:
                                    min_str = min_str[:47] + "..."
                                if len(max_str) > 50:
                                    max_str = max_str[:47] + "..."
                                print(f"  Min: {min_str}")
                                print(f"  Max: {max_str}")
                            else:
                                # For other types (integers, floats, etc.)
                                print(f"  Min: {min_val}")
                                print(f"  Max: {max_val}")
                                
                        except OverflowError:
                            # Handle timestamp overflow by accessing raw values
                            try:
                                # Get the raw underlying values without datetime conversion
                                min_raw = stats.min.as_py() if hasattr(stats.min, 'as_py') else str(stats.min)
                                max_raw = stats.max.as_py() if hasattr(stats.max, 'as_py') else str(stats.max)
                                print(f"  Min (raw): {min_raw}")
                                print(f"  Max (raw): {max_raw}")
                                
                                # Provide context for timestamp values
                                if pa.types.is_timestamp(col_type):
                                    print(f"  Note: Timestamp values outside Python datetime range")
                                    # Show as numeric values for interpretation
                                    if isinstance(min_raw, int) and isinstance(max_raw, int):
                                        days_since_epoch_min = min_raw / (1e9 * 60 * 60 * 24)  # if nanoseconds
                                        days_since_epoch_max = max_raw / (1e9 * 60 * 60 * 24)
                                        print(f"  Min as days since epoch: {days_since_epoch_min:.2f}")
                                        print(f"  Max as days since epoch: {days_since_epoch_max:.2f}")
                            except Exception as inner_e:
                                print(f"  Min: <error: {inner_e}>")
                                print(f"  Max: <error: {inner_e}>")
                                
                        except Exception as e:
                            print(f"  Min: <error: {e}>")
                            print(f"  Max: <error: {e}>")
                    
                    # Distinct count
                    if stats.has_distinct_count:
                        print(f"  Distinct values: {stats.distinct_count:,}")
                    
                else:
                    # Only show no-stats message for the first few row groups to avoid clutter
                    if rg_idx < 3:
                        print(f"\nRow Group {rg_idx}: No statistics available")
                
                total_values += rg_num_rows
            
            # Column summary
            print(f"\n{'─'*40}")
            print(f"SUMMARY for '{column_name}':")
            print(f"{'─'*40}")
            print(f"Total rows examined: {total_values:,}")
            print(f"Total nulls: {total_nulls:,}")
            
            if total_values > 0:
                null_percentage = (total_nulls / total_values) * 100
                non_null_count = total_values - total_nulls
                print(f"Null percentage: {null_percentage:.2f}%")
                print(f"Non-null count: {non_null_count:,}")
            else:
                print("No data available")
            
            if not has_any_stats:
                print("Note: No statistics available in metadata for this column")

def get_quick_column_overview(URL, column_names=None):
    """Get a quick overview of columns without detailed row group breakdown"""
    
    with fsspec.open(URL, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow
        
        if isinstance(column_names, str):
            column_names = [column_names]
        
        valid_columns = [col for col in column_names if col in schema.names]
        
        print(f"=== QUICK OVERVIEW FOR {len(valid_columns)} COLUMN(S) ===")
        
        for column_name in valid_columns:
            col_idx = schema.names.index(column_name)
            col_type = schema.field(column_name).type

            total_nulls = 0
            total_values = 0
            
            # Aggregate statistics across all row groups
            for rg_idx in range(metadata.num_row_groups):
                rg_metadata = metadata.row_group(rg_idx)
                col_metadata = rg_metadata.column(col_idx)
                rg_num_rows = rg_metadata.num_rows
                
                if col_metadata.statistics and col_metadata.statistics.has_null_count:
                    total_nulls += col_metadata.statistics.null_count
                
                total_values += rg_num_rows
            
            null_percentage = (total_nulls / total_values) * 100 if total_values > 0 else 0
            
            print(f"\n{column_name}: {col_type}")
            print(f"  Total rows: {total_values:,}")
            print(f"  Null count: {total_nulls:,} ({null_percentage:.1f}%)")
            print(f"  Non-null: {total_values - total_nulls:,}")

## Posts

In [ ]:
POSTS_URL = "https://bsky-data.leobalduf.com/posts.parquet"

# First, just understand the structure
with fsspec.open(POSTS_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Posts Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Posts Database
Columns: ['did_id', 'rkey', 'created_at', 'languages', 'labels', 'tags', 'embed_type', 'did_id', 'collection', 'rkey', 'embed_external_uri', 'embed_images', 'embed_media', 'embed_video', 'did_id', 'collection', 'rkey', 'did_id', 'collection', 'rkey']
Number of rows: 1290686413
Number of row groups: 10495


In [ ]:
get_quick_column_overview(POSTS_URL, ["did_id", "created_at"])

=== QUICK OVERVIEW FOR 2 COLUMN(S) ===

did_id: int64
  Total rows: 1,290,686,413
  Null count: 0 (0.0%)
  Non-null: 1,290,686,413

created_at: timestamp[us, tz=UTC]
  Total rows: 1,290,686,413
  Null count: 76 (0.0%)
  Non-null: 1,290,686,337


In [27]:
get_detailed_column_stats(POSTS_URL, ["did_id", "created_at"])

=== ANALYZING 2 COLUMN(S) ===

DETAILED STATS FOR 'did_id'

Row Group 0 (Rows: 123,510):
  Compressed size: 11,676 bytes
  Uncompressed size: 988,111 bytes
  Null count: 0 (0.0%)
  Min: 7776
  Max: 34251732

Row Group 1 (Rows: 123,063):
  Compressed size: 12,665 bytes
  Uncompressed size: 984,535 bytes
  Null count: 0 (0.0%)
  Min: 24331
  Max: 34233565

Row Group 2 (Rows: 123,620):
  Compressed size: 15,052 bytes
  Uncompressed size: 988,991 bytes
  Null count: 0 (0.0%)
  Min: 4254
  Max: 34251774

Row Group 3 (Rows: 123,392):
  Compressed size: 16,038 bytes
  Uncompressed size: 987,167 bytes
  Null count: 0 (0.0%)
  Min: 9249
  Max: 34242191

Row Group 4 (Rows: 123,482):
  Compressed size: 17,031 bytes
  Uncompressed size: 987,887 bytes
  Null count: 0 (0.0%)
  Min: 5462
  Max: 34250368

Row Group 5 (Rows: 123,260):
  Compressed size: 9,958 bytes
  Uncompressed size: 986,111 bytes
  Null count: 0 (0.0%)
  Min: 22797
  Max: 34214675

Row Group 6 (Rows: 123,095):
  Compressed size: 13,

## Profiles

In [4]:

PROFILES_URL = "https://bsky-data.leobalduf.com/profiles.parquet"

# First, just understand the structure
with fsspec.open(PROFILES_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Profiles Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Profiles Database
Columns: ['did_id', 'rkey', 'created_at', 'description', 'labels', '/', 'size', 'mimeType', '/', 'size', 'mimeType', 'joined_via_starter_pack', 'additional_fields']
Number of rows: 32170299
Number of row groups: 262


In [7]:
get_quick_column_overview(PROFILES_URL, ["did_id", "created_at", "joined_via_starter_pack"])
get_detailed_column_stats(PROFILES_URL, ["did_id", "created_at"])

=== QUICK OVERVIEW FOR 3 COLUMN(S) ===

did_id: int64
  Total rows: 32,170,299
  Null count: 0 (0.0%)
  Non-null: 32,170,299

created_at: timestamp[us, tz=UTC]
  Total rows: 32,170,299
  Null count: 4,247,596 (13.2%)
  Non-null: 27,922,703

joined_via_starter_pack: extension<arrow.json>
  Total rows: 32,170,299
  Null count: 1,488,624 (4.6%)
  Non-null: 30,681,675
=== ANALYZING 2 COLUMN(S) ===

DETAILED STATS FOR 'did_id'

Row Group 0 (Rows: 122,880):
  Compressed size: 125,140 bytes
  Uncompressed size: 983,071 bytes
  Null count: 0 (0.0%)
  Min: 1
  Max: 130828

Row Group 1 (Rows: 122,880):
  Compressed size: 124,886 bytes
  Uncompressed size: 983,071 bytes
  Null count: 0 (0.0%)
  Min: 130829
  Max: 261540

Row Group 2 (Rows: 122,880):
  Compressed size: 124,887 bytes
  Uncompressed size: 983,071 bytes
  Null count: 0 (0.0%)
  Min: 261541
  Max: 392360

Row Group 3 (Rows: 122,880):
  Compressed size: 124,889 bytes
  Uncompressed size: 983,071 bytes
  Null count: 0 (0.0%)
  Min: 3923

# Cleaning and Preprocessing the data
In both posts and profiles I'm just interested in a few columns:
1) Posts: ["did_id", "created_at"]
2) Profiles: ["did_id", "created_at", "joined_via_starter_pack"]

In [ ]:
def clean_database(input_path, output_path, target_columns=["did_id", "created_at"]):
    """
    Cleans and preprocesses the profiles database using PyArrow only (no pandas).
    """
    
    input_path = os.path.expanduser(input_path)
    output_path = os.path.expanduser(output_path)
    
    print(f"Cleaning profiles database...")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"Target columns: {target_columns}")
    print("=" * 60)
    
    # TODO: Clean null values.
    
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")
    
    try:
        # Step 1: Read only the columns we need
        print("1. Reading input file...")
        read_start = time.time()
        
        profiles_file = pq.ParquetFile(input_path)
        
        # Check which target columns actually exist
        available_columns = profiles_file.schema_arrow.names
        columns_to_read = [col for col in target_columns if col in available_columns]
        
        missing_columns = set(target_columns) - set(available_columns)
        if missing_columns:
            print(f"   Warning: Columns not found: {missing_columns}")
        
        print(f"   Reading {len(columns_to_read)} columns: {columns_to_read}")
        
        # Read the table
        table = profiles_file.read(columns=columns_to_read)
        read_time = time.time() - read_start
        print(f"   Read completed in {read_time:.2f} seconds")
        print(f"   Total records: {table.num_rows:,}")
        
        # Step 2: Clean data using PyArrow compute (memory efficient)
        print("\n2. Cleaning data with PyArrow...")
        clean_start = time.time()
        
        # Get the schema to work with individual columns
        schema = table.schema
        cleaned_arrays = []
        cleaned_schema_fields = []
        
        for i, field in enumerate(schema):
            column_name = field.name
            array = table.column(i)
            
            print(f"   Processing {column_name}...")
            
            if column_name == 'created_at':
                # Ensure timestamp type
                if not pa.types.is_timestamp(field.type):
                    try:
                        # Try to cast to timestamp
                        array = array.cast(pa.timestamp('us', tz='UTC'))
                        print(f"     Cast to timestamp")
                    except Exception as e:
                        print(f"     Warning: Could not cast created_at to timestamp: {e}")
                
            elif column_name == 'did_id':
                # Ensure string type
                if not pa.types.is_string(field.type) and not pa.types.is_integer(field.type):
                    try:
                        array = array.cast(pa.string())
                        print(f"     Cast to string")
                    except Exception as e:
                        print(f"     Warning: Could not cast did_id to string: {e}")
            
            elif column_name == 'joined_via_starter_pack':
                # Handle boolean with nulls
                if pa.types.is_boolean(field.type):
                    # Count nulls before cleaning
                    null_count = array.null_count
                    if null_count > 0:
                        # Fill nulls with False
                        array = array.fill_null(False)
                        print(f"     Filled {null_count:,} nulls with False")
                
            cleaned_arrays.append(array)
            cleaned_schema_fields.append(field.with_type(array.type))
        
        # Create new table with cleaned arrays
        cleaned_table = pa.Table.from_arrays(cleaned_arrays, schema=pa.schema(cleaned_schema_fields))
        
        clean_time = time.time() - clean_start
        print(f"   Cleaning completed in {clean_time:.2f} seconds")
        
        # Step 3: Save the cleaned table
        print("\n3. Saving cleaned data...")
        save_start = time.time()
        
        # Write in batches to avoid memory issues
        pq.write_table(cleaned_table, output_path, 
                      compression='snappy', 
                      row_group_size=100000)  # Smaller row groups for better querying
        
        save_time = time.time() - save_start
        print(f"   Save completed in {save_time:.2f} seconds")
        
        # Step 4: Print summary
        print("\n4. Cleaning Summary:")
        print("=" * 40)
        print(f"   Input records: {table.num_rows:,}")
        print(f"   Output records: {cleaned_table.num_rows:,}")
        print(f"   Final columns: {cleaned_table.schema.names}")
        
        # File size comparison
        input_size = os.path.getsize(input_path) / (1024 * 1024)  # MB
        output_size = os.path.getsize(output_path) / (1024 * 1024)  # MB
        reduction = ((input_size - output_size) / input_size) * 100 if input_size > 0 else 0
        
        print(f"\n   File sizes:")
        print(f"     Input: {input_size:.2f} MB")
        print(f"     Output: {output_size:.2f} MB")
        print(f"     Reduction: {reduction:.1f}%")
        
        total_time = time.time() - read_start
        print(f"\nTotal processing time: {total_time:.2f} seconds")
        
        return output_path
        
    except Exception as e:
        print(f"Error during cleaning: {e}")
        raise

    

In [ ]:
input_path = "/home/ale/Documents/uni/mp/data/raw/profiles.parquet"
output_path = "/home/ale/Documents/uni/mp/data/cleaned/profiles_cleaned.parquet"

clean_database(
    input_path, 
    output_path,
    target_columns=["did_id", "created_at", "joined_via_starter_pack"]
)

Cleaning profiles database...
Input: /home/ale/Documents/uni/mp/data/raw/profiles.parquet
Output: /home/ale/Documents/uni/mp/data/processed/profiles_cleaned.parquet
Target columns: ['did_id', 'created_at', 'joined_via_starter_pack']
1. Reading input file...
   Reading 3 columns: ['did_id', 'created_at', 'joined_via_starter_pack']
   Read completed in 2.99 seconds
   Total records: 32,170,299

2. Cleaning data with PyArrow...
   Processing did_id...
   Processing created_at...
   Processing joined_via_starter_pack...
   Cleaning completed in 0.00 seconds

3. Saving cleaned data...
   Save completed in 12.28 seconds

4. Cleaning Summary:
   Input records: 32,170,299
   Output records: 32,170,299
   Final columns: ['did_id', 'created_at', 'joined_via_starter_pack']

   File sizes:
     Input: 1814.03 MB
     Output: 465.22 MB
     Reduction: 74.4%

Total processing time: 15.27 seconds


'/home/ale/Documents/uni/mp/data/processed/profiles_cleaned.parquet'

In [ ]:
input_path = "/home/ale/Documents/uni/mp/data/raw/chunk_0_posts.parquet"
output_path = "/home/ale/Documents/uni/mp/data/cleaned/chunk_0_posts_cleaned.parquet"

clean_database(
    input_path, 
    output_path,
    target_columns=["did_id", "created_at"]
)

Cleaning profiles database...
Input: /home/ale/Documents/uni/mp/data/raw/chunk_0_posts.parquet
Output: /home/ale/Documents/uni/mp/data/processed/chunk_0_posts_cleaned.parquet
Target columns: ['did_id', 'created_at']
1. Reading input file...
   Reading 2 columns: ['did_id', 'created_at']
   Read completed in 0.51 seconds
   Total records: 5,000,000

2. Cleaning data with PyArrow...
   Processing did_id...
   Processing created_at...
   Cleaning completed in 0.00 seconds

3. Saving cleaned data...
   Save completed in 1.06 seconds

4. Cleaning Summary:
   Input records: 5,000,000
   Output records: 5,000,000
   Final columns: ['did_id', 'created_at']

   File sizes:
     Input: 265.94 MB
     Output: 47.89 MB
     Reduction: 82.0%

Total processing time: 1.57 seconds


'/home/ale/Documents/uni/mp/data/processed/chunk_0_posts_cleaned.parquet'